In [24]:
import os
import time

from dotenv import load_dotenv
from pinecone_setup import Pinecone, ServerlessSpec

load_dotenv()

api_key = os.getenv("PINECONE_API_KEY")

if not api_key:
    raise ValueError("PINECONE_API_KEY is not set")

pc = Pinecone(api_key=api_key)

INDEX_NAME = "contextflow"

# Gemini gemini-embedding-001 produces 3072-dimensional
# embeddings by default.
DIMENSION = 3072

if not pc.has_index(INDEX_NAME):
    print(f"Creating Pinecone index: {INDEX_NAME}")

    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

    # Give Pinecone a moment to initialize the index.
    time.sleep(5)

else:
    print(f"Index already exists: {INDEX_NAME}")

index = pc.Index(INDEX_NAME)

print("Pinecone index is ready!")
print(index.describe_index_stats())

ImportError: cannot import name 'Pinecone' from 'pinecone_setup' (consider renaming 'c:\\Users\\Lenovo\\Documents\\Gen AI\\ContextFlow\\src\\pinecone_setup.py' if it has the same name as a library you intended to import)

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    raise ValueError("OPENROUTER_API_KEY is missing from .env")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

response = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ],
)

print(response.choices[0].message.content)

Hello there! How can I help you today?


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from pinecone import Pinecone

from chunking import chunk_text
from ingestion import load_document


# --------------------------------------------------
# Load environment variables
# --------------------------------------------------
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")
gemini_api_key = os.getenv("GEMINI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY is not set")

if not pinecone_api_key:
    raise ValueError("PINECONE_API_KEY is not set")


# --------------------------------------------------
# Gemini embeddings
# --------------------------------------------------

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=gemini_api_key,
)


# --------------------------------------------------
# Pinecone
# --------------------------------------------------

pc = Pinecone(api_key=pinecone_api_key)

INDEX_NAME = "contextflow"

index = pc.Index(INDEX_NAME)


# --------------------------------------------------
# Load and chunk document
# --------------------------------------------------

document_path = PROJECT_ROOT / "data" / "sample.txt"

text = load_document(str(document_path))

chunks = chunk_text(text)

print(f"Loaded document: {document_path.name}")
print(f"Number of chunks: {len(chunks)}")


# --------------------------------------------------
# Create embeddings
# --------------------------------------------------

print("Creating embeddings...")

vectors = embeddings.embed_documents(chunks)

print(f"Created {len(vectors)} embeddings")


# --------------------------------------------------
# Prepare Pinecone records
# --------------------------------------------------

records = []

for i, (chunk, vector) in enumerate(zip(chunks, vectors)):

    record = {
        "id": f"sample-{i}",
        "values": vector,
        "metadata": {
            "text": chunk,
            "source": "sample.txt",
            "chunk_index": i,
        },
    }

    records.append(record)


# --------------------------------------------------
# Upload to Pinecone
# --------------------------------------------------

print("Uploading vectors to Pinecone...")

index.upsert(vectors=records)

print("Upload complete!")


# --------------------------------------------------
# Show Pinecone statistics
# --------------------------------------------------

stats = index.describe_index_stats()

print("\nPinecone index statistics:")
print(stats)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Loaded document: sample.txt
Number of chunks: 1
Creating embeddings...


GoogleGenerativeAIError: Error embedding content (PERMISSION_DENIED): 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED'}}

In [1]:

from src.verification import verify_answer

evidence = """
Python is commonly used for web development, data analysis,
machine learning, automation, and scripting.
"""

answer = (
    "Python is commonly used for web development "
    "and data analysis."
)

result = verify_answer(answer, evidence)

print("\nVerification result:")
print(result)


ModuleNotFoundError: No module named 'src'

In [ ]:
from dotenv import load_dotenv

load_dotenv()

import os

gemini_api_key = os.getenv("GEMINI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

print("Gemini key loaded:", bool(gemini_api_key))
print("Pinecone key loaded:", bool(pinecone_api_key))

Gemini key loaded: False
Pinecone key loaded: True


In [ ]:
import os

from dotenv import load_dotenv
from xai_sdk import Client
from xai_sdk.chat import user

load_dotenv()

api_key = os.getenv("XAI_API_KEY")

if not api_key:
    raise ValueError("XAI_API_KEY is not set")

print("XAI API key loaded:", True)
print("Key length:", len(api_key))

client = Client(api_key=api_key)

chat = client.chat.create(model="grok-4.7")

chat.append(
    user("Say hello in one sentence.")
)

response = chat.sample()

print(response.content)

XAI API key loaded: True
Key length: 84


_InactiveRpcError: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.PERMISSION_DENIED
	details = "Your newly created team doesn't have any credits or licenses yet. You can purchase those on https://console.x.ai/team/5c432c5f-4667-456f-b490-ef08cc7fb6bb."
	debug_error_string = "PERMISSION_DENIED:Your newly created team doesn't have any credits or licenses yet. You can purchase those on https://console.x.ai/team/5c432c5f-4667-456f-b490-ef08cc7fb6bb."
>

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

key = os.getenv("XAI_API_KEY")

print("Key found:", bool(key))
print("Key length:", len(key) if key else 0)
print("Key preview:", key[:6] + "..." if key else "NOT FOUND")

Key found: True
Key length: 84
Key preview: xai-88...


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

key = os.getenv("XAI_API_KEY")

print("Loaded:", bool(key))
print("Prefix:", key[:4] if key else "NONE")
print("Length:", len(key) if key else 0)

Loaded: True
Prefix: xai-
Length: 84


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

print("Gemini API key loaded:", bool(GOOGLE_API_KEY))

Gemini API key loaded: True


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY,
)

In [ ]:
vector = embeddings.embed_query("What is Retrieval-Augmented Generation?")

print("Embedding worked!")
print("Dimensions:", len(vector))
print("First 5 values:", vector[:5])

Embedding worked!
Dimensions: 3072
First 5 values: [-0.023438238, 0.00724161, -0.00015789657, -0.07040447, -0.012659747]


In [ ]:
from pinecone import Pinecone
import os

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=PINECONE_API_KEY)

index = pc.Index("contextflow")

print(index.describe_index_stats())

{'dimension': 3072,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 4}},
 'total_vector_count': 4,
 'vector_type': 'dense'}


In [ ]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Say hello in one sentence.",
)

print(response.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Say hello in one sentence.",
)

print(response.text)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 6h25m33.583372909s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.8-fla

In [ ]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv(override=True)

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

for model in ["gemini-2.5-flash", "gemini-2.0-flash"]:
    try:
        response = client.models.generate_content(
            model=model,
            contents="Say hello."
        )
        print(model, "WORKS")
        print(response.text)
        break
    except Exception as e:
        print(model, "FAILED")
        print(e)

gemini-2.5-flash FAILED
404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}
gemini-2.0-flash FAILED
404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}


In [ ]:
client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Say hello."
)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 6h24m9.256692603s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.8-flas

In [ ]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Say hello."
)

print(response.text)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 6h15m56.830577282s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-3.8-flash', 'location': 'glob

In [3]:
from src.verification import verify_answer

evidence = """
Python is commonly used for web development, data analysis,
machine learning, automation, and scripting.
"""

answer = (
    "Python is commonly used for web development "
    "and data analysis."
)

result = verify_answer(answer, evidence)

print("\nVerification result:")
print(result)


ModuleNotFoundError: No module named 'src'